# Business purpose — weather availability and leakage

This notebook establishes how an energy portfolio operator should align weather information with business demand, solar and wind sites without leaking realised future conditions into a forecast. It uses the cached **public** Open-Meteo artifact and **simulated** site coordinates; it does not create production model features.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from gridmatch.research.common import project_path
from gridmatch.research.weather import weather_availability, example_weather_tables, save_weather_artifacts

np.random.seed(20260725)
weather = pd.read_parquet(project_path('data', 'processed', 'weather.parquet'))
sites = pd.read_parquet(project_path('data', 'demo', 'sites.parquet'))
assert set(weather['data_origin']) == {'public'}
assert set(sites['data_origin']) == {'simulated'}
assert (pd.to_datetime(weather['issue_time'], utc=True) != pd.to_datetime(weather['valid_time'], utc=True)).all()
availability = weather_availability(weather)
examples = example_weather_tables(weather, sites)
artifact_paths = save_weather_artifacts(weather, sites)
display(availability)
print({name: str(path) for name, path in artifact_paths.items()})

## Issue time, valid time and leakage

`issue_time` is when information became available; `valid_time` is when weather applies. A historical observation describes realised weather. A historical forecast vintage preserves what an operator could actually know at the issue time. Training with realised future weather while evaluating a day-ahead decision leaks information and overstates performance.

In [ ]:
for use_case, table in examples.items():
    print(f'--- {use_case} example ---')
    display(table)
print('Weather missing values:', int(weather.isna().sum().sum()))

## Methodology by site type

- **Demand:** temperature and calendar-aware sensitivity; cloud may proxy lighting/conditions.
- **Solar:** shortwave radiation, cloud and temperature, aligned to the site coordinate.
- **Wind:** wind speed/direction and pressure; hub-height wind is preferable to the current 10 m field.

Coordinate distance is recorded rather than silently treating one grid point as co-located. Missing weather should create availability flags and an explicit fallback hierarchy; it must not be backfilled from future observations.

## Findings, limitations and production implications

**Findings:** the compact artifact supplies 48 public hourly records and the variables needed to demonstrate demand/solar/wind availability tables; issue and valid times remain distinct. **Limitations:** the current Open-Meteo data is ERA5 historical weather, not a complete operational forecast-vintage archive, and one London grid point is used for examples. **Production implications:** archive forecast vintages at issue time, enrich each coordinate, record model/run identifiers and make missing-weather fallback visible before any forecasting model is trained.